Kaggle房价预测

In [ ]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')  # 忽略警告信息

# 相关机器学习库
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

先加载数据：
用 pandas 读取 csv 文件

In [32]:
import os

train_path = r"E:\Kaggle\input\House Prices\train.csv"
test_path  = r"E:\Kaggle\input\House Prices\test.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print(f"训练集形状: {train_df.shape}") 
print(f"测试集形状: {test_df.shape}")

训练集形状: (1460, 81)
测试集形状: (1459, 80)


数据探索：了解数据的基本情况，这有助于我们选择合适的预处理策略（不是必需，但能帮助我们发现数据的问题）

In [8]:
print(train_df.head())

# 查看数据类型的分布(多少列是数值型，多少列是对象/分类型)
print(f"数据型分布：{train_df.dtypes.value_counts()}")

# 查看目标变量 SalePrice 的统计信息
print(f"目标变量 SalePrice 描述统计：{train_df['SalePrice'].describe()}")

   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  
0   2008        WD   

分离特征和目标变量：在训练集中，x_train 是特征矩阵（除Id和SalePrice外所有列），y_train是目标向量；在测试集中，我们保留Id列用于最终的测试

In [11]:
# 训练集： 从train_df中分离出特征和目标变量
y_train = train_df['SalePrice']
x_train = train_df.drop(['Id','SalePrice'], axis=1)
print(f"训练特征 x_train 形状：{x_train.shape}")

# 测试集： 从test_df中分离出特征
test_ids = test_df['Id']  # 保存测试集的Id列
x_test = test_df.drop(['Id'], axis=1)
print(f"测试特征 x_test 形状：{x_test.shape}")


训练特征 x_train 形状：(1460, 79)
测试特征 x_test 形状：(1459, 79)


识别数值列和分别列
对于线性回归我们只能处理数值型数据，所以：
对于数值列，填充缺失值，然后标准化（使不同量纲的特征可比）；对于类别列，填充缺失值，然后进行独热编码（独热编码将每个类别转换成一个0/1的二元列）

In [ ]:
# 找出所有数值型列（int64 和 float64）
numeric_cols = x_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"数值型列数: {len(numeric_cols)}")

# 找出所有对象型列（排除数值型，兼容新版 Pandas）
categorical_cols = x_train.select_dtypes(exclude=['int64', 'float64']).columns.tolist()
print(f"类别型列数: {len(categorical_cols)}")

# 验证：数值列 + 类别列 = 总列数
assert len(numeric_cols) + len(categorical_cols) == x_train.shape[1], "列数不匹配！"

构建数据预处理流水线：
使用 sklearn 的 ColumTransformer 对不爱同类型的数据分别处理；Pipeline 将多个步骤串联起来，确保训练和预测时使用完全相同的变换

In [19]:
# 数值型特征的预处理流水线：填充缺失值 + 标准化
# 第一步：SimpleImputer - 用中位数填充缺失值，选择中位数而非均值，因为中位数不受极端离群值影响
# 第二步：StandarScaler - 标准化（减去均值除以标准差），使每个特征的均值为0、标准差为1，避免大数据特征主导模型
numeric_transformer = Pipeline(steps=[
    ("imputer",SimpleImputer(strategy = 'median')),
    ("scaler",StandardScaler()),
])

In [18]:
# 类别型特征的预处理流水线：填充缺失值 + 独热编码
# 第一步：SimpleImputer - 用最频繁的类别填充缺失值，对于类别变量，用最常见的类别填充是合理的
# 第二步：OneHotEncoder - 独热编码，handle_unknown="ignore"：如果测试集中出现训练集没见过的类别，忽略而非报错；sparse_output=False：返回密集数组，方便后续处理
categorical_transformer = Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False)),
])

In [20]:
# 合并：用 ColumnTransformer 将数值型和类别型特征的预处理流水线合并
# ColumnTransformer 对指定的列自己分别应用相应的变换器，然后对将所有结果按列拼接成一个大的特征矩阵
preprocessor = ColumnTransformer(transformers=[
    ("num",numeric_transformer,numeric_cols),
    ("cat",categorical_transformer,categorical_cols)
])

In [15]:
print("数据预处理流水线构建完成！")

数据预处理流水线构建完成！


构建完整的模型流水线。
将预处理和线性回归模型串联成一个Pipeline，这样做的好处：
防止数据泄露，fit() 时在训练集上学习参数，transform() 时应用到测试集；简化代码，一次 fit() 完成所有步骤；易于交叉验证，整个 Pipeline 可以直接传入 cross_val_score

In [21]:
# 创建完成流水线：预处理 + 线性回归模型
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

训练模型：在训练数据上拟合。fit() 会依次执行：preprocessor.fit_transform(x_train)（学习填充值、编码映射、标准化参数）；model.fit(处理后特征,y_train)（学习线性回归系数）

In [22]:
model_pipeline.fit(x_train,y_train)
print("模型训练完成！")

模型训练完成！


模型评估（交叉验证）：使用 5 折交叉验证来评估模型的泛化能力。R2 分数（决定系数）越接近 1 说明模型拟合越好，负数说明模型比直接使用均值预测还差

In [23]:
# 5 折交叉验证：将训练数据分成5份，轮流用4份训练、1份验证
cv_scores = cross_val_score(
    model_pipeline,x_train,y_train,
    cv = 5,
    scoring = 'r2'  # 使用 R2 分数作为评估指标
)
print(f"各折 R2 分数：{np.round(cv_scores,4)}")
print(f"平均 R2 分数：{cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

各折 R2 分数：[0.8182 0.823  0.8027 0.8879 0.6423]
平均 R2 分数：0.7948 (+/- 0.1633)


In [24]:
# 同时计算均方根误差（RMSE）
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_squared_error

cv_preds = cross_val_predict(model_pipeline,x_train,y_train,cv=5)
rmse = np.sqrt(mean_squared_error(y_train,cv_preds))
print(f"交叉验证的均方根误差（RMSE）：{rmse:.4f}")
print(f"RMSE 占均值的比例：{rmse/y_train.mean()*100:.2f}%")

交叉验证的均方根误差（RMSE）：36414.2569
RMSE 占均值的比例：20.13%


In [26]:
# 查看线性回归的系数
# 获取经预处理后的所有特征名称
preprocessor_fitted = model_pipeline.named_steps['preprocessor']
# 获取独热编码后的所有特征名称
ohe = preprocessor_fitted.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = ohe.get_feature_names_out(categorical_cols).tolist()
all_feature_names = numeric_cols + cat_feature_names
print(f"总特征数：{len(all_feature_names)}")
print(f"前10个特征名称：{all_feature_names[:10]}")

总特征数：287
前10个特征名称：['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2']


In [27]:
# 取出回归系数
coefficients = model_pipeline.named_steps['model'].coef_
coef_df = pd.DataFrame({
    '特征': all_feature_names,
    '系数': coefficients
}).sort_values('系数',key=abs,ascending=False)
print("前10个绝对值最大的特征及其系数：")
print(coef_df.head(10))

前10个绝对值最大的特征及其系数：
                   特征             系数
124  RoofMatl_ClyTile -605775.036857
101   Condition2_PosN -193660.577873
126  RoofMatl_Membran  153007.526614
127    RoofMatl_Metal  123647.369400
131  RoofMatl_WdShngl  114743.000139
248     GarageQual_Ex  101765.174113
253     GarageCond_Ex  -92765.647596
100   Condition2_PosA   79922.198732
123    RoofStyle_Shed   74913.558634
102   Condition2_RRAe  -63358.993954


对测试集进行预测，使用训练好的模型对测试集进行预测，predict() 会自动限制性预处理变换，再调用回归器预测

In [28]:
test_predictions = model_pipeline.predict(x_test)
print(f"预测完成，共{len(test_predictions)}条测试数据的房价预测结果。")

预测完成，共1459条测试数据的房价预测结果。


In [29]:
# 查看预测值的统计信息
print(f"\n预测 SalePrice 描述统计：")
print(f"  最小值:  ${test_predictions.min():,.0f}")
print(f"  最大值:  ${test_predictions.max():,.0f}")
print(f"  均值:    ${test_predictions.mean():,.0f}")
print(f"  中位数:  ${np.median(test_predictions):,.0f}")


预测 SalePrice 描述统计：
  最小值:  $916
  最大值:  $753,562
  均值:    $179,293
  中位数:  $160,750


In [30]:
# 检查是否有不合理的赋值预测
negative_count = (test_predictions < 0).sum()
if negative_count > 0:
    print(f"\n[警告] 发现 {negative_count} 个负值预测，将其设为 0")
    test_predictions[test_predictions < 0] = 0

生成提交文件

In [ ]:
# 创建提交 DataFrame
submission = pd.DataFrame({
    "Id":        test_ids,
    "SalePrice": test_predictions
})

# 保存为 CSV 文件（不保存行索引）
output_path = r"C:\Users\36263\Desktop\submission.csv"
submission.to_csv(output_path, index=False)
print(f"提交文件已保存至: {output_path}")
print(f"文件形状: {submission.shape}")  # 预期 (1459, 2)

# 预览前10行
print("\n--- 提交文件前10行预览 ---")
print(submission.head(10).to_string(index=False))